In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

#import dash
import dash

#import math
import math

#Prevent callback errors from popping up after receiving no data
from dash.exceptions import PreventUpdate

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
#df.drop(columns=['_id'],inplace=True)

#Create a new age based on weeks
df['age_years'] = (df['age_upon_outcome_in_weeks'] / 52).round(0)

#Creating a dictionary key/value for "professional" column names
included_columns_map = {
    'name': 'Name',
    'animal_type': 'Type of Animal',
    'breed': 'Breed',
    'sex_upon_outcome': 'Gender',
    'reproduction_status': 'Spayed/Neutered',
    'color': 'Color',
    'age_years': 'Age (Years)',
    'location_lat': 'Latitude',
    'location_long': 'Longitude'
}

#Converting age to a numeric type to sort properly
df['age_upon_outcome'] = pd.to_numeric(df['age_upon_outcome'], errors='coerce')

#Separating the Gender from their statuses and storing them elsewhere for later
df['reproduction_status'] = df['sex_upon_outcome'].str.split().str[0]
df.loc[df['reproduction_status'].isin(['Male', 'Female']), 'reproduction_status'] = 'Intact'

#Clean the Gender to strictly be M or F
df['sex_upon_outcome'] = df['sex_upon_outcome'].str.split().str[-1]

#Handles "Unknown" status for both
df.loc[df['sex_upon_outcome'] == 'Unknown', 'reproduction_status'] = 'Unknown'

#For the range slider, grab the max age so we don't have to hard code the max age
highest_age_years = math.ceil(df['age_upon_outcome_in_weeks'].max() / 52)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' # replace with your own image
with open(image_filename, 'rb') as f:
    encoded_image = base64.b64encode(f.read()).decode('ascii')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    html.Center(html.B(html.H1('CS-340 Dashboard - Michael Peck'))),
    html.Hr(),
    
    #Main Container
    html.Div(style={'display': 'flex', 'flex-direction': 'row'}, children=[
        
        #Sidebar section for filters
        html.Div(
            style={
                'width': '10%', 
                'padding': '20px', 
                'fontSize': '20px', # This sets a base size for everything inside
                'background-color': '#f8f9fa'
            },
            children=[
                html.Img(
                    src=f'data:image/png;base64,{encoded_image}',
                    style={'height': '200px', 'width': 'auto'} # Optional styling
                ),
                html.H2("Filter Results", style={'fontSize': '24px', 'fontWeight': 'bold', 'marginTop': '10px'}),
                html.Label("Select Animal Type:", style={'fontSize': '20px', 'fontWeight': 'bold', 'marginTop': '10px'}),
                html.Br(),
                html.Br(),
                dcc.Checklist(
                    id='filter-type',
                    options=[
                        {'label': 'Cat', 'value': 'A'},
                        {'label': 'Dog', 'value': 'B'},
                        {'label': 'Bird', 'value': 'C'},
                        {'label': 'Livestock', 'value': 'D'},
                        {'label': 'Other', 'value': 'E'},
                    ],
                    value=[],#Default value
                    inputStyle={"transform": "scale(1.5)", "margin-right": "10px"}, 
                    labelStyle={
                        'display': 'flex',      # Flex ensures the box and text align vertically
                        'align-items': 'center', 
                        'fontSize': '20px',     # Set your desired text size here
                        'marginBottom': '10px'
                    },
                ),
                html.Br(),
                html.Label("Rescue Type:", style={'fontSize': '20px', 'fontWeight': 'bold', 'marginTop': '10px'}),
                html.Br(),
                html.Br(),
                dcc.RadioItems(
                    id='rescue-filter',
                    options=[
                        {'label': 'Water', 'value': 'water'},
                        {'label': 'Mountain/Wilderness', 'value': 'mount_wild'},
                        {'label': 'Disaster', 'value': 'disaster'},
                        {'label': 'Show All', 'value': 'all'}
                    ],
                    value='all',
                    labelStyle={
                        'display': 'flex',      # Flex ensures the box and text align vertically
                        'align-items': 'center', 
                        'fontSize': '22px',     # Set your desired text size here
                        'marginBottom': '10px'
                    },
                    inputStyle={"margin-right": "10px"}
                ),
                html.Br(),
                html.Label("Select Gender:", style={'fontSize': '20px', 'fontWeight': 'bold', 'marginTop': '10px'}),
                html.Br(),
                dcc.Checklist(
                    id='filter-gender',
                    options=[
                        {'label': 'Male', 'value': 'Male'},
                        {'label': 'Female', 'value': 'Female'},
                        {'label': 'Unknown', 'value': 'Unknown'}
                    ],
                    value=[],
                    inputStyle={"transform": "scale(1.5)", "margin-right": "10px"},
                    labelStyle={
                        'display': 'flex',      # Flex ensures the box and text align vertically
                        'align-items': 'center', 
                        'fontSize': '22px',     # Set your desired text size here
                        'marginBottom': '10px',
                        'marginTop': '15px'
                    },
                ),
                html.Br(),
                html.Label("Filter by Age (Years):", style={'fontSize': '20px', 'fontWeight': 'bold', 'marginTop': '10px'}),
                html.Div(
                    style={'width': '90%', 'margin': 'auto', 'paddingTop': '20px'}, 
                    children=[
                        dcc.RangeSlider(
                            id='age-slider',
                            min=0,
                            max=highest_age_years,
                            step=1,
                            value=[0, highest_age_years],
                            marks={i: {'label': str(i), 'style': {'fontSize': '14px'}} for i in range(0, highest_age_years + 1, 5)}
                        )
                    ]
                ),
            ]
        ),
        
    html.Div(style={'width': '90%', 'padding': '20px'}, children=[
        #Adding the Input for the Search Bar
        #Adding an Input for a search bar
        dcc.Input(
            id="search-input",
            type="text",
            placeholder="Search by Name or Breed of Animal",
            style={"width": "450px", "marginRight": "10px", 'fontSize': '16px', 'marginBottom': '20px'}
        ),
        
        #Button for the trigger
        html.Button(
            "Search",
            id="search-btn",
            n_clicks=0,
            style={'fontSize': '16px', 'margin-right': '20px'}
        ),
        #Button for the reset for the search
        html.Button(
            "Clear Search",
            id="reset-btn",
            n_clicks=0,
            style={'fontSize': '16px', 'margin-right': '20px'}
        ),
        dash_table.DataTable(id='datatable-id',
            columns=[
                {"name": included_columns_map.get(i, i), "id": i,"type": "numeric" if i == 'age_upon_outcome' else "text", "deletable": False, "selectable": True} for i in included_columns_map
                    ],
            data=df.to_dict('records'),
                             
            #Trying something with cells versus radio buttons
            cell_selectable=True,
            selected_cells=[],
            style_cell={'outline': 'none',
                        'fontSize': '16px',
                       'padding' :'10px'
            },
            style_header={
                'fontSize': '18px',
                'fontWeight': 'bold',
                'backgroundColor': 'rgb(230, 230, 230)',
            },
                             
            #Changing the whole row to blue with one click
            style_data_conditional=[
                {
                    'if': {'state': 'active'},
                    'backgroundColor': '#D2F3FF',
                    'border': '1px solid blue',
                },
                {
                    'if': {'state': 'selected'},
                    'backgroundColor': '#D2F3FF',
                    'border': '1px solid blue',
                }
            ],
            

            #Set up the features for your interactive data table to make it user-friendly for your client
            sort_action="native",
            filter_action="native",
            page_action="native",
            page_size=10,
        ),
        html.Br(),
        html.Hr(),
        #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
        html.Div(className='row',
             style={'display' : 'flex'},
                 children=[
            html.Div(style={'flex': '1'},
                id='graph-id',
                className='col s12 m6',
                ),
            html.Div(style={'flex': '1'},
                id='map-id',
                className='col s12 m6',
                )
            ])
        ])
    ])
])

#Every time a search/reset happens, the data gets muddled. This function should clear it up
def clean_data(data_list):
    if not data_list:
        return []
    
    temp_df = pd.DataFrame.from_records(data_list)
    
    # Apply your Sex/Status split logic
    if 'sex_upon_outcome' in temp_df.columns:
        # Create Status
        temp_df['reproduction_status'] = temp_df['sex_upon_outcome'].str.split().str[0]
        temp_df.loc[temp_df['reproduction_status'].isin(['Male', 'Female']), 'reproduction_status'] = 'Intact'
        
        # Clean Gender
        temp_df['sex_upon_outcome'] = temp_df['sex_upon_outcome'].str.split().str[-1]
        
        # Sync Unknowns
        temp_df.loc[temp_df['sex_upon_outcome'] == 'Unknown', 'reproduction_status'] = 'Unknown'
        
    #Handle secondary age
    if 'age_upon_outcome_in_weeks' in temp_df.columns:
        #recalculate age again
        temp_df['age_years'] = (temp_df['age_upon_outcome_in_weeks'] / 52).round(0)

    # Handle Age
    if 'age_upon_outcome' in temp_df.columns:
        temp_df['age_upon_outcome'] = pd.to_numeric(temp_df['age_upon_outcome'], errors='coerce')
        
    return temp_df.to_dict('records')

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [Output('datatable-id', 'data'),
    Output('search-input','value'),
    Output('filter-type','value'),
     Output('filter-gender','value'),
    Output('rescue-filter', 'value')],
    [Input('search-btn', 'n_clicks'), #Performs the query
     Input('reset-btn', 'n_clicks'), #Resets the query
     Input('filter-type', 'value'), #Performs a filter
     Input('filter-gender','value'),
     Input('age-slider', 'value'), #Handles the age slider
     Input('rescue-filter', 'value')],
    [State('search-input', 'value')]
)
def perform_search_filter(search_clicks, reset_clicks, filter_values, gender_values, age_range, rescue_type, search_value):
    # Determine what triggered the callback
    ctx = dash.callback_context
    if not ctx.triggered:
        raise PreventUpdate
        
    triggered_id = ctx.triggered[0]['prop_id'].split('.')[0]
    
    all_values = []
    all_genders = []
    
    # 1. Handle Reset Button
    if triggered_id == 'reset-btn':
        results = shelter.read({})
        return [clean_data(results), "", all_values, all_genders, 'all']
    query = {
        "age_upon_outcome_in_weeks": {
            "$gte": age_range[0] * 52, 
            "$lte": age_range[1] * 52
        }
    }
    
    if filter_values:
        mapping = {'A': 'Cat', 'B': 'Dog', 'C': 'Bird', 'D': 'Livestock', 'E': 'Other'}
        selected_types = [mapping[v] for v in filter_values if v in mapping]
        if selected_types:
            query['animal_type'] = {'$in': selected_types}
            
    if gender_values:
        # This turns ['Male', 'Female'] into "Male|Female"
        gender_pattern = "|".join([rf"\b{g}\b" for g in gender_values])
        
        query['sex_upon_outcome'] = {
            '$regex': gender_pattern, 
            '$options': 'i'
        }
            
    if rescue_type == 'water':
        water_regex = 'Labrador|Chesa|Newfoundland'
        query.update({
            "breed": {'$regex': water_regex, "$options": "i"},
            "sex_upon_outcome": "Intact Female",
            # Widening age to 13 weeks (3 months) up to 520 weeks (10 years)
            "age_upon_outcome_in_weeks": {"$gte": 13, "$lte": 520} 
        })        
        
    elif rescue_type == 'mount_wild':
        mountain_regex = 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler'
        query.update({
            "animal_type": "Dog",
            "breed": {'$regex': mountain_regex, "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }) 
        
    elif rescue_type == 'disaster':
        disaster_regex = 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler'
        query.update({
            "breed": {"$regex": disaster_regex, "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }) 

    #Handle Search Button
    if triggered_id =='search-btn' and search_value:
        # Use only text columns for regex search to avoid errors with numeric columns
        search_cols = ['name', 'breed', 'animal_type', 'sex_upon_outcome', 'color']
        search_fields = [{field: {"$regex": search_value, "$options": "i"}} for field in search_cols]
        
        query = {"$or": search_fields}
        results = shelter.read(query)
        return [clean_data(results), search_value, all_values, all_genders, 'all']
    
    results = shelter.read(query)
    return [clean_data(results), search_value, filter_values, gender_values, rescue_type]
    
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    #Ensure data is there before displaying
    if viewData is None or len(viewData) ==0:
        return []
    
    #Convert table data into pandas Dataframe
    dff = pd.DataFrame.from_dict(viewData)
    
    mapping = {'A': 'Cat', 'B': 'Dog', 'C': 'Bird', 'D': 'Livestock', 'E': 'Other'}
    if 'animal_type' in dff.columns:
        dff['animal_type'] = dff['animal_type'].map(mapping).fillna(dff['animal_type'])
    
    #Create a sunburst chart that divides first into cat vs dog, and then into breeds
    fig = px.sunburst(
        dff,
        path=['animal_type', 'breed'],
        title='Breeds by Animal Type'
    )
    
    #Show labels for small slices
    fig.update_traces(textinfo="label+percent entry")
    return [
        dcc.Graph(figure=fig)
    ]
    
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'active_cell'),
     Input('datatable-id', 'selected_columns')]
)
def update_styles(active_cell, selected_columns):
    if active_cell is None:
        return []
    #Forcing the active cell to match the row colors of light blue
    styles = [
        {
            'if': {'state': 'active'},
            'backgroundColor': '#D2F3FF',
            'border': '1px solid blue'
        },
        {
            'if': {'state': 'selected'},
            'backgroundColor': '#D2F3FF',
            'border': '1px solid blue'
        },
    ]
    if active_cell:
        #adds the blue row highlight
        styles.append({
            'if': {'row_index': active_cell['row']},
            'backgroundColor': '#D2F3FF',
            'border': '1px solid blue'
        })
    return styles


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
    Input('datatable-id', "active_cell")])
def update_map(viewData, active_cell):
#FIXME Add in the code for your geolocation chart
    if viewData is None or active_cell is None or len(active_cell)== 0:
        #Do not callback function if no data is received
        raise PreventUpdate
        #Default view of Austin if no results
        return[dl.Map(style={'width': "1000px", "height": "500px"},
                      center=[30.75, -97.48], zoom=10,
                      children=[dl.TileLayer()])]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    #Attempting to grab the row the currently selected cell is in
    row = active_cell['row']
    #row = index[0]
    
    #Pull data by name, but by column count
    lat=dff.iloc[row]['location_lat']
    lon=dff.iloc[row]['location_long']
    breed=dff.iloc[row]['breed']
    name=dff.iloc[row]['name']
        
    #Austin TX is at [30.75, -97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[lat, lon], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   #Marker with tool tip and popup
                   #13 and 14 define grid coordinates
                   #4 defines the breed
                   #9 defines the name
                   dl.Marker(position=[lat, lon],
                             children=[
                                 dl.Tooltip(breed),
                                 dl.Popup([
                                     html.H1("Animal Name"),
                                     html.H2(name)
                                 ])
                             ])
               ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://egorespect-zebrabeast-3000.codio.io/proxy/8050/
